In [2]:
import ultralytics
from ultralytics import YOLO
import os
import cv2
import time
import torch
import random
import shutil
import tqdm
from sklearn.metrics import precision_score, recall_score


In [3]:
a=[0,0,0]
b=[1,0,1]
print(precision_score(a,b))

0.0


In [4]:
print("CUDA Available: " + str(torch.cuda.is_available()))
print("Torch CUDA Version: " + str(torch.version.cuda))
# Check to make sure CUDA is available and does not say "None"
ultralytics.utils.checks.collect_system_info()

Ultralytics YOLOv8.1.27 🚀 Python-3.11.8 torch-2.2.1 CUDA:0 (NVIDIA GeForce RTX 4090 Laptop GPU, 15919MiB)
Setup complete ✅ (32 CPUs, 31.0 GB RAM, 2402.2/3372.6 GB disk)

OS                  Linux-6.8.0-101-generic-x86_64-with-glibc2.35
Environment         Jupyter
Python              3.11.8
Install             pip
RAM                 31.02 GB
CPU                 AMD Ryzen 9 7945HX with Radeon Graphics
CUDA                12.1

matplotlib          ✅ 3.8.3>=3.3.0
opencv-python       ✅ 4.9.0.80>=4.6.0
pillow              ✅ 10.2.0>=7.1.2
pyyaml              ✅ 6.0.1>=5.3.1
requests            ✅ 2.31.0>=2.23.0
scipy               ✅ 1.12.0>=1.4.1
torch               ✅ 2.2.1>=1.8.0
torchvision         ✅ 0.17.1>=0.9.0
tqdm                ✅ 4.67.1>=4.64.0
psutil              ✅ 5.9.8
py-cpuinfo          ✅ 9.0.0
thop                ✅ 0.1.1-2209072238>=0.1.1
pandas              ✅ 2.2.1>=1.1.4
seaborn             ✅ 0.13.2>=0.11.0


In [3]:
CURR_DIR = os.getcwd()
WORKSPACE_DIR = os.path.dirname(CURR_DIR)
DATASETS_DIR = WORKSPACE_DIR + '/datasets/true_pos'
DATA_YAML = WORKSPACE_DIR + '/data.yaml'
CURR_RUN = 'ims_2024_day4_run1_vimba_front_frameskip_5_filtered'
TEST_PATH = DATASETS_DIR + f'/{CURR_RUN}/images/'

In [12]:
CURR_MODEL_PATH = WORKSPACE_DIR + '/models/yolo_model_files/epoch95.pt'
model = YOLO(CURR_MODEL_PATH)
data_dir = CURR_DIR + '/../data/'
TEST_PATH = DATASETS_DIR + f'/{CURR_RUN}/images/'
# Run predictions on a directory of images
results = model.predict(source=TEST_PATH, save=False, verbose=False)

# Extract predictions
predictions = []
for result in results:
    print(result)
    if result.boxes:
        conf = result.boxes.conf.cpu().numpy().tolist()
        ids = result.boxes.cls.cpu().numpy().tolist()
        seg_masks = result.masks.data

        for i in range(len(result.boxes)):
            x1, y1, x2, y2 = result.boxes[i].xyxy.cpu().numpy().tolist()[0] 
            width = x2-x1
            height = y2-y1
            area = width * height
            
            predictions.append({
                'image_id': result.path,
                'bbox_area': area,
                'confidence': conf[i],
                'class_id': ids[i]
            })
    else:
        predictions.append({
                'image_id': result.path,
                'bbox_area': 0,
                'confidence': 0,
                'class_id': 0
            })


ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: ultralytics.engine.results.Masks object
names: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair

In [ ]:
print(len(predictions))

1386


: 

In [9]:
#def test_precision(model : YOLO, test_results_path: os.PathLike) -> None:

# TEST_PATH = DATASETS_DIR + f'/{CURR_RUN}/images/'
LABEL_PATH = DATASETS_DIR + f'/{CURR_RUN}/labels/'
# # Run predictions on a directory of images
# results = model.predict(source=TEST_PATH, save=False, verbose=True)

# # Extract predictions
# predictions = []
# for result in results:
#     print(result)
#     if len(result.boxes) > 0:
#         conf = result.boxes.conf.cpu().numpy().tolist()
#         ids = result.boxes.cls.cpu().numpy().tolist()


#         for i in range(len(result.boxes)):
#             x1, y1, x2, y2 = result.boxes[i].xyxy.cpu().numpy().tolist()[0] 
#             width = x2-x1
#             height = y2-y1
#             area = width * height
            
#             predictions.append({
#                 'image_id': result.path,
#                 'bbox_area': area,
#                 'confidence': conf[i],
#                 'class_id': ids[i]
#             })
#     else:
#         predictions.append({
#                 'image_id': result.path,
#                 'bbox_area': 0,
#                 'confidence': 0,
#                 'class_id': 0
#             })

y_true = []
y_preds = []
# extract labels
for file in os.listdir(TEST_PATH):
    file_path = os.path.join(TEST_PATH, file)
    label_file_path = os.path.join(LABEL_PATH, file_path[-16:-4]) + '.txt'
    print(label_file_path)
    #print(output[0].masks)
    #save_path = os.path.join(base_save_path, file)

    # grab true label (1 if the label file exists, 0 if the label file doesn't exist)
    if not os.path.exists(label_file_path):
        y_true.append(0)
    else:
        # append number of detections
        with open(label_file_path, 'r') as file:
            lines = file.readlines()
            line_count = len(lines)
            print(line_count)
            y_true.extend([1 for i in range(line_count)])
        

print(len(y_true))
print(len(predictions))

/home/annabelng/Desktop/YOLOv8-Fine-Tune/datasets/true_pos/ims_2024_day4_run1_vimba_front_frameskip_5_filtered/labels/frame_000443.txt
1
/home/annabelng/Desktop/YOLOv8-Fine-Tune/datasets/true_pos/ims_2024_day4_run1_vimba_front_frameskip_5_filtered/labels/frame_000083.txt
2
/home/annabelng/Desktop/YOLOv8-Fine-Tune/datasets/true_pos/ims_2024_day4_run1_vimba_front_frameskip_5_filtered/labels/frame_000394.txt
2
/home/annabelng/Desktop/YOLOv8-Fine-Tune/datasets/true_pos/ims_2024_day4_run1_vimba_front_frameskip_5_filtered/labels/frame_000272.txt
2
/home/annabelng/Desktop/YOLOv8-Fine-Tune/datasets/true_pos/ims_2024_day4_run1_vimba_front_frameskip_5_filtered/labels/frame_000183.txt
2
/home/annabelng/Desktop/YOLOv8-Fine-Tune/datasets/true_pos/ims_2024_day4_run1_vimba_front_frameskip_5_filtered/labels/frame_000380.txt
2
/home/annabelng/Desktop/YOLOv8-Fine-Tune/datasets/true_pos/ims_2024_day4_run1_vimba_front_frameskip_5_filtered/labels/frame_000433.txt
1
/home/annabelng/Desktop/YOLOv8-Fine-Tune/

In [18]:
def test_boundingbox_model(model : YOLO, test_results_path: os.PathLike) -> None:
    """
    Test the fine-tuned model on test images and save the results.

    Parameters:
        model (YOLO): The fine-tuned YOLO model.
        test_results_path (os.PathLike): The path to save the test results.

    Returns:
        None

    """

    bounding_box_results = {}
    size_thresholds = [30,40,50,60,70,80]

    # Make sure the test save path exists
    if not os.path.exists(test_results_path):
        os.makedirs(test_results_path)

    TEST_PATH = DATASETS_DIR + f'/{CURR_RUN}/images/'
    for size_threshold in size_thresholds:
        false_pos = 0
        false_neg = 0
        total_positives = 0
        true_neg = 0
        
        base_save_path = os.path.join(test_results_path, str(size_threshold))

        if not os.path.exists(base_save_path):
            os.makedirs(base_save_path)
        
        # Inference fine-tuned model on test images and save results
        for file in os.listdir(TEST_PATH):
            valid_box_found = False
            file_path = os.path.join(TEST_PATH, file)
            output = model.predict(file_path)
            save_path = os.path.join(base_save_path, file)
            #print(save_path)

            if len(output[0].boxes) == 0:
                true_neg += 1
            else:
                # add bounding box filter threshold
                x1, y1, x2, y2 = output[0].boxes[0].xyxy[0].tolist()
                width = x2-x1
                height = y2-y1
                area = width * height

                if area < size_threshold:
                    true_neg += 1
        
                else:
                    false_pos += 1
                    annotated_img = output[0].plot()
                    save_status = cv2.imwrite(save_path, annotated_img)
                    if not save_status:
                        raise RuntimeError("failed to save")
                
        total_preds = true_neg + false_pos
        if total_preds > 0:
            precision = true_neg / (total_preds)
        # Log results with folder path
        log_message = (
            f'\nCurr run: {CURR_RUN}\n'
            f'Bounding box size threshold: {size_threshold} \n Precision: {precision:.4f}\n'
        )
        with open(CURR_DIR + '/../results/precision.txt', 'a') as log_file:
            log_file.write(log_message)
        
        print(log_message.strip())  # Print to console as well

        bounding_box_results[size_threshold] = [true_neg, false_pos, precision]

    return bounding_box_results

In [19]:
def test_conf_model(model : YOLO, test_results_path: os.PathLike) -> None:
    """
    Test the fine-tuned model on test images and save the results.

    Parameters:
        model (YOLO): The fine-tuned YOLO model.
        test_results_path (os.PathLike): The path to save the test results.

    Returns:
        None

    """
    precisions = []
    confidence_thresholds = [0.4]

    # Make sure the test save path exists
    if not os.path.exists(test_results_path):
        os.makedirs(test_results_path)

    TEST_PATH = DATASETS_DIR + f'/{CURR_RUN}/images/'
    for conf in confidence_thresholds:
        false_pos = 0
        false_neg = 0
        total_positives = 0
        true_neg = 0
        
        base_save_path = os.path.join(test_results_path, 'confidence_boundbox', str(conf * 100))

        if not os.path.exists(base_save_path):
            os.makedirs(base_save_path)
        
        # Inference fine-tuned model on test images and save results
        for file in os.listdir(TEST_PATH):
            valid_box_found = False
            file_path = os.path.join(TEST_PATH, file)
            output = model.predict(file_path)
            save_path = os.path.join(base_save_path, file)
            #print(save_path)

            #print(output[0].boxes)
            if len(output[0].boxes) == 0:
                true_neg += 1
            else:
                # Filter boxes by confidence > 50%
                high_conf_boxes = [box for box in output[0].boxes if box.conf > conf]

                if len(high_conf_boxes) == 0:
                    true_neg += 1
                else:
                    # add bounding box filter threshold
                    x1, y1, x2, y2 = output[0].boxes[0].xyxy[0].tolist()
                    width = x2-x1
                    height = y2-y1
                    area = width * height

                    if area < 50:
                        true_neg += 1
        
                    else:
                        false_pos += 1
                        annotated_img = output[0].plot()
                        save_status = cv2.imwrite(save_path, annotated_img)
                        if not save_status:
                            raise RuntimeError("failed to save")



                # if len(high_conf_boxes) > 0:
                #     false_pos += 1
                #     annotated_img = output[0].plot()
                #     save_status = cv2.imwrite(save_path, annotated_img)
                #     if not save_status:
                #         raise RuntimeError("failed to save")

            
                
        total_preds = true_neg + false_pos
        if total_preds > 0:
            precision = true_neg / (total_preds)
        # Log results with folder path
        log_message = (
            f'\nCurr run: {CURR_RUN}\n'
            f'Bounding box size threshold: {conf} \n Precision: {precision:.4f}\n'
        )
        with open(CURR_DIR + '/../results/precision.txt', 'a') as log_file:
            log_file.write(log_message)
        
        print(log_message.strip())  # Print to console as well
        precisions.append(precision)
        #bounding_box_results[size_threshold] = [true_neg, false_pos, precision]

    return precisions

In [20]:
def test_confidence_model(model : YOLO, test_results_path: os.PathLike) -> None:
    """
    Test the fine-tuned model on test images and save the results.

    Parameters:
        model (YOLO): The fine-tuned YOLO model.
        test_results_path (os.PathLike): The path to save the test results.

    Returns:
        None

    """
    confidence_thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
    # Make sure the test save path exists
    if not os.path.exists(test_results_path):
        os.makedirs(test_results_path)

    for conf in confidence_thresholds:
        # Inference fine-tuned model on test images and save results
        for file in os.listdir(TEST_PATH):
            file_path = os.path.join(TEST_PATH, file)
            output = model.predict(file_path)
            save_path = os.path.join(test_results_path, file)
            #print(output[0].boxes)
            if len(output[0].boxes) == 0:
                true_neg += 1
            else:
                # Filter boxes by confidence > 50%
                high_conf_boxes = [box for box in output[0].boxes if box.conf > conf]
                if len(high_conf_boxes) > 0:
                    false_pos += 1

                    # Save annotated image with bounding boxes
                    annotated_img = output[0].plot()
                    cv2.imwrite(save_path, annotated_img)

        total_preds = true_neg + false_pos
        if total_preds > 0:
            precision = true_neg / (total_preds)
        # Log results with folder path
        log_message = (
            f'\nCurr run: {CURR_RUN}\n'
            f'Confidence threshold: {conf} \n Precision: {precision:.4f}\n'
        )
        with open(CURR_DIR + '/../results/precision.txt', 'a') as log_file:
            log_file.write(log_message)
        
        print(log_message.strip())  # Print to console as well
    return true_neg, false_pos, precision

In [ ]:
def test_precision_true_pos(model : YOLO, test_results_path: os.PathLike) -> None:
    """
    Test the fine-tuned model on test images and save the results.

    Parameters:
        model (YOLO): The fine-tuned YOLO model.
        test_results_path (os.PathLike): The path to save the test results.

    Returns:
        None

    """

    bounding_box_results = {}
    confidence_thresholds = [0.2,0.3,0.4,0.5,0.6,0.7]

    # Make sure the test save path exists
    if not os.path.exists(test_results_path):
        os.makedirs(test_results_path)

    TEST_PATH = DATASETS_DIR + f'/{CURR_RUN}/images/'
    LABEL_PATH = DATASETS_DIR + f'/{CURR_RUN}/labels/'
    print(LABEL_PATH)
    for confidence in confidence_thresholds:
        # create arrays for true pos and predictions to calculate precision
        y_true = []
        preds = [] 

        base_save_path = os.path.join(test_results_path, str(confidence))

        if not os.path.exists(base_save_path):
            os.makedirs(base_save_path)
        
        # Inference fine-tuned model on test images and save results
        for file in os.listdir(TEST_PATH):
            curr_y_true = []
            curr_preds = []

            file_path = os.path.join(TEST_PATH, file)
            label_file_path = os.path.join(LABEL_PATH, file_path[-16:-4]) + '.txt'
            print(label_file_path)
            output = model.predict(file_path)
            print(output[0].masks)
            save_path = os.path.join(base_save_path, file)

            # grab true label (1 if the label file exists, 0 if the label file doesn't exist)
            if not os.path.exists(label_file_path):
                curr_y_true.append(0)
                line_count = 1
            else:
                with open(label_file_path, 'r') as file:
                    lines = file.readlines()
                    line_count = len(lines)
                    print(line_count)
                    curr_y_true.extend([1 for i in range(line_count)])

            # no boxes means no prediction was detected 
            if len(output[0].boxes) == 0:
                curr_preds.append(0)
            else:
                high_conf_boxes = [box for box in output[0].boxes if box.conf > confidence and box.cls == 2]
                curr_preds.extend([1 for i in range(max(len(high_conf_boxes), line_count))])
            
            if len(curr_preds) > len(curr_y_true):
                curr_y_true.extend([0 for i in range(len(curr_preds) - len(curr_y_true))])
            else:
                curr_preds.extend([0 for i in range(len(curr_y_true) - len(curr_preds))])

            preds.extend(curr_preds)
            y_true.extend(curr_y_true)
        #print(len(y_true))
        print(preds)
        precision = precision_score(y_true, preds)
        recall = recall_score(y_true, preds)
        # Log results with folder path
        log_message = (
            f'\nCurr run: {CURR_RUN}\n'
            f'Confidence threshold: {confidence} \nPrecision: {precision:.4f}\nRecall: {recall:.4f}\n'
        )
        with open(CURR_DIR + '/../results/precision.txt', 'a') as log_file:
            log_file.write(log_message)
        
        print(log_message.strip())  # Print to console as well
    return y_true, preds
    # return None, None

In [ ]:
#initialize model
runs = ['ims_2024_day6_run1_vimba_rear_filtered', 'ks_2024_day15_run3_left']
#runs = ['ims_2024_day4_run1_vimba_front_frameskip_5_filtered']
for run in runs:
    CURR_RUN = run

    CURR_MODEL_PATH = WORKSPACE_DIR + '/models/yolo_model_files/epoch95.pt'
    model = YOLO(CURR_MODEL_PATH)
    data_dir = CURR_DIR + '/../data/'

    # initialize test results
    test_results_path = CURR_DIR + f'/../results/output_imgs/{CURR_RUN}'
    y_true, preds = test_precision_true_pos(model, test_results_path)


/home/annabelng/Desktop/YOLOv8-Fine-Tune/datasets/true_pos/ims_2024_day4_run1_vimba_front_frameskip_5_filtered/labels/
/home/annabelng/Desktop/YOLOv8-Fine-Tune/datasets/true_pos/ims_2024_day4_run1_vimba_front_frameskip_5_filtered/labels/frame_000443.txt

image 1/1 /home/annabelng/Desktop/YOLOv8-Fine-Tune/datasets/true_pos/ims_2024_day4_run1_vimba_front_frameskip_5_filtered/images/frame_000443.PNG: 800x1056 3 cars, 3.7ms
Speed: 2.2ms preprocess, 3.7ms inference, 0.8ms postprocess per image at shape (1, 3, 800, 1056)
ultralytics.engine.results.Masks object with attributes:

data: tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        [[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0.

In [47]:
y_true

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,


In [ ]:
#initialize model
CURR_MODEL_PATH = WORKSPACE_DIR + '/models/yolo_model_files/epoch95.pt'
model = YOLO(CURR_MODEL_PATH)
data_dir = CURR_DIR + '/../data/'

# initialize test results
test_results_path = CURR_DIR + f'/../results/annotated_imgs/{CURR_RUN}'
true_neg, false_pos, precision = test_model(model, test_results_path)



image 1/1 /home/annabelng/Desktop/YOLOv8-Fine-Tune/datasets/true_negative/ims_2024_day4_run1_vimba_right/images/frame_002783.PNG: 800x1056 (no detections), 3.8ms
Speed: 2.4ms preprocess, 3.8ms inference, 0.2ms postprocess per image at shape (1, 3, 800, 1056)
ultralytics.engine.results.Boxes object with attributes:

cls: tensor([], device='cuda:0')
conf: tensor([], device='cuda:0')
data: tensor([], device='cuda:0', size=(0, 6))
id: None
is_track: False
orig_shape: (772, 1032)
shape: torch.Size([0, 6])
xywh: tensor([], device='cuda:0', size=(0, 4))
xywhn: tensor([], device='cuda:0', size=(0, 4))
xyxy: tensor([], device='cuda:0', size=(0, 4))
xyxyn: tensor([], device='cuda:0', size=(0, 4))

image 1/1 /home/annabelng/Desktop/YOLOv8-Fine-Tune/datasets/true_negative/ims_2024_day4_run1_vimba_right/images/frame_003408.PNG: 800x1056 (no detections), 3.4ms
Speed: 2.1ms preprocess, 3.4ms inference, 0.2ms postprocess per image at shape (1, 3, 800, 1056)
ultralytics.engine.results.Boxes object wit

KeyboardInterrupt: 

In [68]:
# ims day 5
print(f'True neg: {true_neg}, False_Pos: {false_pos}, Precision: {precision}')

True neg: 7186, False_Pos: 134, Precision: 0.9816939890710382


In [17]:
# ims day 6 run 1 rear
print(f'True neg: {true_neg}, False_Pos: {false_pos}, Precision: {precision}')

True neg: 808, False_Pos: 1, Precision: 0.9987639060568603


In [71]:
# ims day 6 run 2
print(f'True neg: {true_neg}, False_Pos: {false_pos}, Precision: {precision}')

True neg: 2975, False_Pos: 93, Precision: 0.9696870925684485


In [ ]:
# ims day 6 run 1: 808 no detections, 1 car detection, 99.87% precise
# ims day 6 run 2: 2975 no detections, 98 car detections, 95.97% precise 
# ims day 5: 7186 no detections, 134 car detections, 98.17% precise